In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset

from fluidFlow.dit import DiT
from fluidFlow.trainer import Trainer
from fluidFlow.flow_matching import create_flow_matching

try:
    import kaggle
    kaggle_available = True
except Exception as e:
    print(e)
    print("Kaggle api won't work, either donwload the data or follow the steps to authenticate with your kaggle account")
    kaggle_available = False


DATA_DIR = "../data"

/home/d.ramos/miniconda/envs/nibbler/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0708 18:00:06.006000 2023900 site-packages/torch/utils/cpp_extension.py:140] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Flash attention 4 enabled ⚡
Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
name 'exit' is not defined
Kaggle api won't work, either donwload the data or follow the steps to authenticate with your kaggle account


In [2]:
def get_data_from_kaggle():

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        
        kaggle.api.competition_download_files(
            'sharp-raptor-challenge',
            path=tmp_path
        )
        
        zip_file = next(tmp_path.glob('*.zip'))
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(tmp_path)
        
        df_train = pd.read_csv(tmp_path / 'train.csv')
        df_test = pd.read_csv(tmp_path / 'test.csv')
        airfoil_df = pd.read_csv(tmp_path / 'airfoil_coordinates.csv')
        validation_ids = pd.read_csv(tmp_path / 'validation_cases.csv')

    return df_train, df_test, airfoil_df, validation_ids

def get_data(data_dir):
    data_dir = Path(data_dir)
    df_train = pd.read_csv(data_dir / 'train.csv')
    df_test = pd.read_csv(data_dir / 'test.csv')
    airfoil_df = pd.read_csv(data_dir / 'airfoil_coordinates.csv')
    validation_ids = pd.read_csv(data_dir / 'validation_cases.csv')

    return df_train, df_test, airfoil_df, validation_ids

In [3]:
if kaggle_available:
    df_train, df_test, airfoil_df, validation_ids = get_data_from_kaggle()
else:
    df_train, df_test, airfoil_df, validation_ids = get_data(DATA_DIR)

In [4]:
num_points = len(airfoil_df)
num_points

13862

In [5]:
flow_conditions = (
    df_train[["case_id", "mach", "aoa"]]
    .drop_duplicates()
    .sort_values("case_id")
)
flow_conditions_test = (
    df_test[["case_id", "mach", "aoa"]]
    .drop_duplicates()
    .sort_values("case_id")
)

# One row per flow condition, one column per surface point
cp_matrix = (
    df_train
    .pivot(index="case_id", columns="point_id", values="cp")
    .sort_index()
)

assert (flow_conditions.case_id.values == cp_matrix.index.values).all()

X = torch.tensor(
    flow_conditions[["mach", "aoa"]].values,
    dtype=torch.float32,
)

Y = torch.tensor(
    cp_matrix.values,
    dtype=torch.float32,
).unsqueeze(1)

X_test = torch.tensor(
    flow_conditions_test[["mach", "aoa"]].values,
    dtype=torch.float32,
)

X.shape, Y.shape, X_test.shape

(torch.Size([85, 2]), torch.Size([85, 1, 13862]), torch.Size([15, 2]))

In [6]:
X_mean, X_std = X.mean(dim=0), X.std(dim=0)
Y_mean, Y_std = Y.mean(), Y.std()
X = (X - X_mean) / X_std
Y = (Y - Y_mean) / Y_std

X_test = (X_test - X_mean) / X_std

X_mean, X_std

(tensor([0.8943, 2.5065]), tensor([0.3537, 1.4537]))

Since the number of points needs to be divisible by the patch size, we need to add padding to enable larger patch sizes.

In [7]:
pad_length = 13872 # divisible by 16
Y = F.pad(Y, (0, pad_length - num_points), mode='constant', value=0)

In [8]:
dataset_train = TensorDataset(
    Y, X
)
# test dataset with dummy cp values
dataset_test = TensorDataset(
    torch.zeros((X_test.shape[0], 1, num_points)), X_test
)

In [9]:
model = DiT(
    depth=6,
    hidden_size=512,
    patch_size=16,
    num_heads=16,
    input_size=Y.shape[-1], # dataset image size
    cond_dim=2,
    class_dropout_prob=0.2,
    in_channels=1,
    use_swiglu=True,
    # qk_norm=True, # when bf16 training
    attn_type="physics",  # window, linear, vanilla
    mlp_ratio=2.5,
)

Creating 1D DiT
Creating DiT with 867 patches.


In [10]:
flow_matching = create_flow_matching(
    neural_net=model,
    input_size=Y.shape[-1],
    cond_scale=2.0,
    sampling_method="euler",
    num_sampling_steps=100,
)

In [11]:
results_folder = './results'
train_steps = 100000
trainer = Trainer(
    flow_matching,
    dataset=dataset_train,
    train_batch_size=16,
    train_lr=1e-4,
    train_num_steps=train_steps,  # total training steps
    gradient_accumulate_every=1,  # gradient accumulation steps
    ema_decay=0.995,  # exponential moving average decay
    # amp=True,     # turn on mixed precision for faster training and reduced memory usage
    # mixed_precision_type='bf16',
    results_folder=results_folder,  # folder to save results to
    save_and_sample_every=20000,
    eta_min_scheduler=4e-6,
    max_grad_norm=1.0,
    # use_cpu=True,
    use_muon=True,
    compile_model=True,
    split_batches=True
)

Compiling model...
Model compiled


In [ ]:
trainer.train()

In [ ]:
trainer.ema.ema_model.eval()
# this will sample with multiple gpus, if available
samples, seqs = trainer.eval_model(dataset_test, batch_size=16, use_autocast=True)

samples = (samples.cpu() * Y_std) + Y_mean
samples = samples[..., :num_points]
torch.save(samples, f"{results_folder}/test_predictions_ema.pt")
samples = samples.flatten()
df_test["cp"] = samples
df_sumbission = df_test[["id", "cp"]]
print(df_sumbission.head())
df_sumbission.to_csv("submission.csv", index=False)
